In [ ]:
import pandas as pd

from public_power_backend.constants import DATA_DIR, OUTPUT_DIR

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
ious_df = pd.read_parquet(OUTPUT_DIR / "data_warehouse/pudl_utilities.parquet")

In [ ]:
pse_df = pd.read_csv("pse_utility_summary.csv").drop(columns=["Unnamed: 0"])

In [ ]:
col_map = {
    "Utility Number": "utility_id_eia",
    "Utility Name": "utility_name_eia",
    "Median Cost Burden": "residential_cost_burden_median_percent",
    "Median Cost Burden Poverty<2.5": "residential_cost_burden_below_2.5_median_percent",
}

In [ ]:
pse_df = pse_df.rename(columns=col_map)

In [ ]:
pse_df = pse_df.drop_duplicates()

In [ ]:
ious_df = ious_df.merge(
    pse_df[
        [
            "utility_id_eia",
            "residential_cost_burden_median_percent",
            "residential_cost_burden_below_2.5_median_percent",
        ]
    ],
    how="left",
    on="utility_id_eia",
    validate="1:1",
)

In [ ]:
ious_df

Get 5 year averages of SAIDI, CAIDI, SAIFI

In [ ]:
reliability_df = pd.read_parquet(
    "s3://pudl.catalyst.coop/stable/core_eia861__yearly_reliability.parquet",
    dtype_backend="pyarrow",
)

In [ ]:
reliability_df

In [ ]:
reliability_df = reliability_df[
    [
        "utility_id_eia",
        "report_date",
        "standard",
        "saidi_w_major_event_days_minus_loss_of_service_minutes",
        "caidi_w_major_event_days_minus_loss_of_service_minutes",
        "saifi_w_major_event_days_minus_loss_of_service_customers",
    ]
]

In [ ]:
reliability_df = reliability_df.sort_values(
    by=["utility_id_eia", "report_date", "standard"]
).drop_duplicates(subset=["utility_id_eia", "report_date"], keep="first")

In [ ]:
min_report_year = 2019
max_report_year = 2023

In [ ]:
reliability_df = reliability_df[
    (reliability_df.report_date.dt.year >= min_report_year)
    & (reliability_df.report_date.dt.year <= max_report_year)
]

In [ ]:
reliability_df

In [ ]:
avg_metrics = (
    reliability_df.groupby(["utility_id_eia"])[
        [
            "saidi_w_major_event_days_minus_loss_of_service_minutes",
            "caidi_w_major_event_days_minus_loss_of_service_minutes",
            "saifi_w_major_event_days_minus_loss_of_service_customers",
        ]
    ]
    .mean()
    .reset_index()
)

In [ ]:
ious_df = ious_df.merge(avg_metrics, how="left", on="utility_id_eia", validate="1:1")

In [ ]:
ious_df

Get nameplate capacity

In [ ]:
eia_gens = pd.read_parquet(
    "s3://pudl.catalyst.coop/stable/out_eia__yearly_generators_by_ownership.parquet",
    dtype_backend="pyarrow",
)

In [ ]:
eia_gens

In [ ]:
mask = (eia_gens.report_date.dt.year == 2023) & (~eia_gens.utility_id_eia.isnull())

In [ ]:
util_cap = eia_gens[mask]

In [ ]:
util_cap["utility_owned_capacity_mw"] = (
    util_cap["capacity_mw"] * util_cap["fraction_owned"]
)

In [ ]:
util_cap_by_fuel = (
    util_cap.groupby(["utility_id_eia", "fuel_type_code_pudl"])[
        ["utility_owned_capacity_mw"]
    ]
    .sum()
    .reset_index()
)

In [ ]:
df_wide = util_cap_by_fuel.pivot_table(
    index="utility_id_eia",
    columns="fuel_type_code_pudl",
    values=["utility_owned_capacity_mw"],
    aggfunc="sum",  # or "first" if there’s no duplication
)

In [ ]:
df_wide.columns = [f"{cls}_{metric}" for metric, cls in df_wide.columns]

In [ ]:
ious_df = ious_df.merge(df_wide, how="left", on="utility_id_eia", validate="1:1")

In [ ]:
ious_df

In [ ]:
fossil_fuels = ["coal", "gas", "nuclear", "oil", "waste"]
fossil_cols = [fuel + "_utility_owned_capacity_mw" for fuel in fossil_fuels]

In [ ]:
fossil_cols

In [ ]:
ious_df["total_non_renewable_capacity_mw"] = ious_df[fossil_cols].sum(axis=1)

Grab Emily Grubert projected GHG Emissions

In [ ]:
ghg_df = pd.read_excel(
    DATA_DIR / "input/grubert_projected_ghg.xlsx",
    sheet_name="All Utilities GHG Results",
    header=1,
)

In [ ]:
ghg_df = ghg_df.rename(
    columns={"Utility Number": "utility_id_eia", 2050: "2050_projected_ghg_emissions"}
)

In [ ]:
ghg_df[ghg_df.utility_id_eia.duplicated(keep=False)]

In [ ]:
ghg_df = ghg_df[ghg_df.utility_id_eia != 88888]

In [ ]:
ious_df = ious_df.merge(
    ghg_df[["utility_id_eia", "2050_projected_ghg_emissions"]],
    how="left",
    on="utility_id_eia",
    validate="1:1",
)

In [ ]:
ious_df

In [ ]:
ious_df.to_csv("utilitygenic_harms.csv")